<a href="https://colab.research.google.com/github/m97j/cwie/blob/main/train/Lora_npc_persona_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---
# 📘 **Persona_Chat_Engine** project's **main model fine tuning** notebook
---

## 프로젝트 개요

- **목표**: 단일 디코더 LLM이 동시에
  - 자연어 LM 응답 생성,
  - 상태 변화 예측(Delta: 연속값, tanh로 [-1,1]),
  - 행동 플래그 예측(Flag: 멀티라벨, 시그모이드)를 수행하고,

  RAG로 세계관/설명 지식을 주입해 맥락·서사 정합성을 높인다.

- **학습 형태**: QLoRA 위에 LoRA 어댑터 미세조정.
  - 메모리·대역폭을 줄이면서 성능 손실을 최소화
  - Colab Plus A100(40GB VRAM)에서 안정적으로 학습.

- **Base 모델**: Qwen2.5-3B-Instruct
  - RoPE, GQA, RMSNorm, SwiGLU로 긴 문맥·속도·안정성 균형.
  - 한국어 포함 multi-lingual 대화 적합.

- **데이터 구조**: \<SYS> → \<RAG> → \<PLAYER_STATE> → \<CTX> → \<PLAYER> → \<STATE> → \<NPC> 고정.
  - <STATE> 이전 맥락을 세 태스크가 공유(공동표현 학습).
  - 학습 시 <RAG>는 비워두고 추론 시 주입.

---

## 프로젝트 목표 기반 학습 형태 선정 근거

 - 프로젝트 목표는 단일 모델이 **대화(LM) + 상태 변화(Delta) + 행동 플래그(Flag)**를 동시에 예측하는 것이다. 이 경우, full fine-tuning은 모든 weight를 업데이트하므로 다음 문제가 생긴다:

    1. **자원 한계**: Colab Plus A100 VRAM(40GB)로도 3B~7B full fine-tuning은 효율이 낮음  
    2. **Overfitting 위험**: 전체 파라미터 변경은 작은 데이터셋일 때 과적합 확률이 높음  
    3. **배포 유연성 부족**: 모델마다 full weight 저장 → 10~20GB 이상

    이 문제를 LoRA를 선택하여 해결했다.

- ### LoRA 선택 이유 [표현력 보존]
  - LoRA(+$\,$QLoRA) 선택 근거는 “저랭크 보정만으로도 표현력 유지”라는 점이다.

    - Transformer의 특정 가중치 최적값 $[W^\star]$가 베이스 $[W_0]$로부터 변화 $[\Delta W = W^\star - W_0]$를 갖는다고 하자.    
    Gradient Descent로 얻는 업데이트 $\Delta W$는 실험적으로 랭크가 낮다
      - (gradient subspace의 effective rank는 보통 10~100 수준).

      SVD로 분해하면:
      $[\Delta W = U \Sigma V^\top]$

      상위 $r$개의 특이값만 유지한 최적 저랭크 근사(에카르트-영 정리):

      - $[\Delta W_r = U_r \Sigma_r V_r^\top,\quad r \ll \min(d_{\text{out}}, d_{\text{in}})]
      $는

        $[\|\Delta W - \Delta W_r\|_F]$를 최소화한다.

      LoRA의 $[BA]$는 바로 이 $[\Delta W_r]$를 구현할 수 있다
      - (적당한 $B=U_r \Sigma_r^{1/2},\ A=\Sigma_r^{1/2} V_r^\top$ 선택으로 동일한 랭크-$r$ 근사 가능).  

      따라서 $[W' = W_0 + \alpha BA]$로 $W^\star$에 근접하는 해를 표현할 수 있다.  
      딥러닝 학습에서 관찰되는 “유효 그라디언트 서브스페이스”의 차원 $r$이 상대적으로 작기 때문에, 충분히 작은 $r$로도 성능 손실이 작다.


- QLoRA는 $W_0$를 4bit로 저장(nf4 quant)하되 연산은 bf16으로 수행해 양자화 오차를 최소화한다.

---

## Base Model 구조와 선택 근거
- 기반 구조는 **Transformer Decoder-only**다.

- ### 1. Transformer 디코더 아키텍처

  - 디코더 블록 계산

    - **Self-Attention (Causal)**:  
    입력 임베딩을 $[X \in \mathbb{R}^{T \times d}]$라 하고, 레이어 인덱스를 $[\ell]$이라 하자.

      - $[Q = XW_Q,\quad K = XW_K,\quad V = XW_V]$
      - $[Z = \frac{QK^\top + M}{\sqrt{d_k}}]$  
        - (여기서 $M$은 $t$ 이후 토큰을 차단하는 causal mask: $M_{ij}=-\infty$ for $j>i$)
      - $[P = \mathrm{softmax}(Z)]$
      - $[Y = PV]$

      이후 잔차·정규화·FFN(SwiGLU):
      
    - **Residual + RMSNorm + SwiGLU FFN**:  
    $[
    U = \text{SwiGLU}(YW_1 + b_1)W_2 + b_2
    ]$  
    출력은 다시 residual로 합쳐 다음 레이어로.

    - 최종 레이어 출력은 $[H^{(L)}]$로 표기한다.

- ### 2. Transformer 디코더 아키텍처 기반 Base model의 확장 아키텍처

  - RoPE(상대 위치 인코딩)

    - RoPE는 각 채널의 2D-쌍에 대해 회전 행렬을 적용한다. 채널 인덱스 $[i]$의 각 2D-쌍에 대해 위치 $[t]$에서

      - $[\theta_i = \omega^{i} \cdot t]$ (주파수 기저 $[\omega]$는 사전 정의)
      - $[\mathrm{RoPE}(q_{t,2i:2i+2}) = R(\theta_i)\, q_{t,2i:2i+2}]$, $[\mathrm{RoPE}(k_{t,2i:2i+2}) = R(\theta_i)\, k_{t,2i:2i+2}]$

    - 여기서 $[R(\theta)]$는 2D 회전. 이로써 $[q\cdot k]$ 내적이 상대적 위치 차이에만 의존하는 성질을 갖는다(긴 문맥에서 강건).

  - GQA(Grouped Query Attention)

    - GQA는 다수의 Query 헤드를 소수의 KV 그룹에 매핑한다. 즉, $[K,V]$는 그룹 수 $[G]$로 축소 저장되고, 여러 $[Q]$가 동일 그룹의 $[K,V]$를 공유해 캐시 메모리를 절약한다. 수식은 표준 MHA와 동일하지만, $[K,V]$가 그룹 인덱스에 의해 공유된다.


- **Qwen2.5-3B-Instruct**는 Transformer 디코더 아키텍처 기본 구조에 RoPE(상대 위치 인코딩), GQA(Grouped Query Attention)로 KV 캐시 효율 최적화, RMSNorm·SwiGLU로 학습 안정성을 강화한다.  
목표(대화·맥락 유지·다국어 지원)와 자원(A100, 3B 크기) 조건에서 최적.
- ### 따라서 해당 모델을 base model로 결정하게 되었다.

---

## 모델 학습 pipeline

### "**데이터 전처리 → Forward → Loss → Backward → Optimization → Validation**"

- ### 1. Forward의 내부 동작과 Attention 흐름

  **전처리**에서 `<STATE>` 위치를 마킹, `<NPC>` 이전 라벨은 -100으로 마스킹한다.

  **Forward** 시:
  - **LM 경로(토큰 생성)**: $H^{(L)}$ 전체를 사용해 LM Head에서 각 토큰의 logits를 산출한다.
    - 훈련: teacher forcing. 타겟 $[y_t]$에 대해 $[\log p(y_t|x_{\le t})]$를 계산.
    - 추론: 자기회귀 생성. 첫 단어는 입력 프롬프트(<STATE> 포함)에 대한 $[H^{(L)}]$로 예측. 이후 $t{+}1$번째 단어는 $t$까지의 출력 토큰을 포함한 시퀀스로 재계산(또는 KV 캐시 사용).
  - **Delta/Flag 경로**: `<STATE>` 위치의 hidden state를 평균 풀링해 $h_*$를 얻고, 이를 Delta/Flag head에 각각 전달한다.
    - $[\mathcal{S}]$: 입력에서 <STATE> 토큰의 인덱스 집합(보통 1개).
    - $[h_* = \frac{1}{|\mathcal{S}|}\sum_{t\in\mathcal{S}} H^{(L)}_t]$ (없으면 마지막 토큰 사용).
    - $[\hat{\delta}=\tanh(W_\Delta h_*) \in [-1,1]^2]$, $[\hat{p}=\sigma(W_F h_*) \in [0,1]^C]$.

  즉, LM은 전체 시퀀스 hidden의 시간적 전개(자기회귀), Delta/Flag는 \<STATE> 위치 표현의 “요약”에 매핑된다.

  이 시점에서 우리는 세 가지 예측 결과를 모두 확보했고, 이제 이를 기반으로 **Loss 계산 단계**로 넘어간다.

- ### 2. 손실 함수 설계와 가중치의 이론적 근거
  Forward 출력에서 나온 LM logits, Delta 예측, Flag 확률을 각각의 태스크 특성에 맞춰 손실로 변환한다.

  #### 각 손실의 선택 이유

  - **LM(CE)**: 토큰-조건부 확률의 로그우도를 직접 최대화 → 언어 생성 정확도에 직결.
  - **Delta(Huber)**: MSE의 민감성과 MAE의 강건성을 절충. 대화 데이터의 노이즈/표현 다양성으로 인한 outlier에 강건.
  - **Flag(BCEWithLogits)**: 독립 멀티라벨 시그모이드 확률을 직접 최적화. 클래스 불균형은 $[\text{pos_weight}]$로 보정.

  세 손실은 다음과 같이 가중합된다.
  $$[
  \mathcal{L} = 0.34\,\mathcal{L}_{\text{LM}} + 0.33\,\mathcal{L}_{\Delta} + 0.33\,\mathcal{L}_{\text{Flag}}
  ]$$

  이 가중치는 각 태스크의 그라디언트 규모를 균형 있게 유지해 안정적인 학습 방향을 제공한다.  
  이제 이 총손실 $\mathcal{L}$을 출발점으로 **Backward 단계**에서 기울기를 전파한다.

  #### 가중합과 Gradient Balancing

  - 총손실:
  $[
  \mathcal{L} = \lambda_{\text{LM}} \mathcal{L}_{\text{LM}} + \lambda_\Delta \mathcal{L}_\Delta + \lambda_{\text{Flag}} \mathcal{L}_{\text{Flag}},\quad (\lambda_{\text{LM}},\lambda_\Delta,\lambda_{\text{Flag}})=(0.34,0.33,0.33)
  ]$

    - **정리(가중합과 파레토 최적성)**: 각 태스크 손실 $[\mathcal{L}_i]$가 볼록이고 공역이 동일하다면, 임의의 파레토 최적점은 어떤 양의 가중치 벡터 $[\lambda]$에 대한 가중합 최소화 문제의 해로 표현된다.  
      비록 본 문제는 비볼록이지만, 지역 파레토 정지점에서는 $[\sum_i \lambda_i \nabla \mathcal{L}_i = 0]$를 만족하는 $[\lambda]$가 존재(필요조건).

    - **균형 선택의 직관**: $[\nabla \mathcal{L}] = \sum_i \lambda_i \nabla \mathcal{L}_i]$.  
      동일 스케일(노름)로 각 그라디언트를 반영하려면 $[\lambda_i]$는 각 태스크의 기대 그라디언트 노름을 보정하도록 설정되어야 한다.  
      초기에는 동일 가중(0.34/0.33/0.33)로 시작해 학습 안정성과 공정한 수렴을 유도한다(필요시 GradNorm류의 동적 조정으로 확장 가능).

    - **스케치 증명**: 가중합 목적의 stationary point에서 $[\nabla \mathcal{L}=\sum_i \lambda_i \nabla \mathcal{L}_i=0]$.  
      만약 특정 태스크 그라디언트가 과도하게 크면, 그 태스크가 전체 하강 방향을 지배하여 다른 태스크의 손해로 이어질 수 있다.  
      균형 가중은 이 위험을 줄이고 파레토 전선 근방에서 공정한 감소 방향을 택할 확률을 높인다.
- ### 3. Backward: 미분 경로와 LoRA 업데이트
  총손실 $\mathcal{L}$을 로짓에 대해 미분하면 CE의 기본식  

  $[
  \frac{\partial \mathcal{L}_{\text{LM}}}{\partial z_t} = s_t - y_t
  ]$
  가 나온다.

    - #### Softmax-CE의 기본 미분

      로짓 $[z_t]$, softmax $[s_t=\mathrm{softmax}(z_t)]$, 원-핫 $[y_t]$일 때,  
      $[\frac{\partial \mathcal{L}_{\text{LM}}}{\partial z_t} = s_t - y_t]$

    이는 LM 경로의 그라디언트를 로짓에서 hidden으로 전파하는 출발점이다.

  이 그라디언트는 Attention 경로를 따라 $Q,K,V$ 및 각 프로젝션 가중치로 전파된다.  
  Softmax Jacobian, $Z$에 대한 미분, 그리고 $Q,K$에 대한 기울기 계산은 다음과 같이 흘러간다.

  #### i) 어텐션의 미분 흐름
  - 앞서 $[Y = PV]$, $[P=\mathrm{softmax}(Z)]$, $[Z=(QK^\top+M)/\sqrt{d_k}]$라 할 때, 상류에서 $[\frac{\partial \mathcal{L}}{\partial Y}]$가 주어지면

    - $[\frac{\partial \mathcal{L}}{\partial V} = P^\top \frac{\partial \mathcal{L}}{\partial Y}]$
    - $[\frac{\partial \mathcal{L}}{\partial P} = \frac{\partial \mathcal{L}}{\partial Y} V^\top]$
    - softmax의 Jacobian을 이용해
      - $[\frac{\partial \mathcal{L}}{\partial Z} = P \odot \left( \frac{\partial \mathcal{L}}{\partial P} - (P \odot \frac{\partial \mathcal{L}}{\partial P})\mathbf{1}^\top \right)]$
      - 관용적으로 $[\frac{\partial \mathcal{L}}{\partial Z} = P \circ (G - \mathrm{row\_sum}(P \circ G))]$로 표기 가능. $[G=\frac{\partial \mathcal{L}}{\partial P}]$
    - $[\frac{\partial \mathcal{L}}{\partial Q} = \frac{1}{\sqrt{d_k}} \left(\frac{\partial \mathcal{L}}{\partial Z}\right) K]$
    - $[\frac{\partial \mathcal{L}}{\partial K} = \frac{1}{\sqrt{d_k}} \left(\frac{\partial \mathcal{L}}{\partial Z}\right)^\top Q]$  
    여기서 $[Q=XW_Q,\ K=XW_K,\ V=XW_V]$이므로

    - $[\frac{\partial \mathcal{L}}{\partial W_Q} = X^\top \frac{\partial \mathcal{L}}{\partial Q}],\quad \frac{\partial \mathcal{L}}{\partial W_K} = X^\top \frac{\partial \mathcal{L}}{\partial K},\quad \frac{\partial \mathcal{L}}{\partial W_V} = X^\top \frac{\partial \mathcal{L}}{\partial V}]$

    이 미분 경로는 LM 경로뿐 아니라, Delta/Flag head까지 거슬러 올라가 $[H^{(L)}]$→각 어텐션 가중치→프로젝션 가중치로 전파된다.

  #### ii) LoRA 파라미터의 미분

  - LoRA는 특정 프로젝션(예: $W_Q,W_K,W_V,W_O$ 또는 FFN의 $W_1,W_2$)을 $[W' = W_0 + \alpha BA]$로 대체한다.  
  $[\mathcal{L}]$을 $[W']$로 미분한 $[\frac{\partial \mathcal{L}}{\partial W'}]$가 주어지면,

    - 체인룰로 $[\frac{\partial \mathcal{L}}{\partial A} = \alpha\, B^\top \frac{\partial \mathcal{L}}{\partial W'}]$
    - $[\frac{\partial \mathcal{L}}{\partial B} = \alpha\, \frac{\partial \mathcal{L}}{\partial W'} A^\top]$


  이렇게 계산된 그라디언트는 이제 **최적화 단계**에서 파라미터를 갱신하는 데 사용된다.

- ### 4. Optimization: AdamW와 수렴 논리
  AdamW는 1·2차 모멘트 추정치로 학습률을 조정하며, weight decay로 과적합을 억제한다.

    - **AdamW** 업데이트:  
      $[
      m_t=\beta_1 m_{t-1}+(1-\beta_1)g_t,\quad v_t=\beta_2 v_{t-1}+(1-\beta_2)g_t^2
      ]$  
      - 여기서 $g_t$는 LoRA 파라미터 $A,B$에 대한 그라디언트.

      $[
      \hat{m}_t=\frac{m_t}{1-\beta_1^t},\quad \hat{v}_t=\frac{v_t}{1-\beta_2^t},\quad
      \theta\leftarrow \theta - \eta \frac{\hat{m}_t}{\sqrt{\hat{v}_t}+\epsilon} - \eta\lambda \theta
      ]$
      

    - **LoRA만 업데이트**: $[\theta=\{A,B\}]$로 제한해 용량을 제어(정규화 효과).  
      이는 sharp-minima로의 과도한 수렴을 억제하고, 일반화에 유리한 평평한 해를 찾는데 도움.

  이 과정은 모델이 sharp minima 대신 평평한 minima로 수렴하도록 돕고,  
  다음 **Validation 단계**에서 성능이 실제로 향상되었는지 확인한다.

- ### 5. Validation
  Epoch를 단순히 늘리는 것은 $\hat{R}$ (훈련 손실)은 줄여도 일반화 성능 $R$을 보장하지 않습니다.  
  따라서 ppl, $R^2_{\Delta}$, $\text{F1}_{\text{macro,Flag}}$를 합성한


  $[
  J_{\text{dev}} = -\log(\text{ppl}) + R^2_\Delta + \text{F1}_{\text{macro,Flag}}
  ]$


  을 dev 세트에서 최적화 목표로 사용한다.  
  이 복합 지표는 세 태스크의 균형 성능을 반영하며, best composite 시점에 per-class threshold를 조정해 Flag 예측을 개선한다.


  #### Validation이 Epoch 증가보다 유리한 수식적 근거

  - **일반화 경계(직관)**: 테스트 위험 $[R(f)] \le \hat{R}(f) + \mathcal{C}(f,\mathcal{H},n)]$로 경계.  
    여기서 $[\hat{R}]$는 훈련 손실, $[\mathcal{C}]$는 용량(라데마허 복잡도 등).  
    Epoch 증가로 $[\hat{R}]$는 감소하지만, 복잡도 증가(또는 샤프 미니마 수렴)로 $[R]$이 증가 가능(과적합).

  - **Composite dev 목적**:
    $[
    J_{\text{dev}} = -\log(\text{ppl}) + R^2_\Delta + \text{F1}_{\text{macro,Flag}}
    ]$
    세 태스크에 대한 일반화 성능을 합성해 최적화. 단일 지표 대비 **다목적 일반화**에 더 근접.

  - **스케치 증명**: 멀티태스크 일반화 목적 $[\vec{R}(f)]$에 대해, 스칼라화 $[\phi(\vec{R})=a\cdot (-\log \text{ppl}) + b\cdot R^2 + c\cdot \text{F1}]$로 surrogate를 최소화하면 파레토 전선 근방의 점을 선택한다(연속·단조 가정).  
    Epoch 증가만으로는 $[\phi]$가 항상 감소하지 않으며, dev 기반 조기선택은 $[\phi]$를 직접 개선하는 선택을 제공.

  - **Threshold 튜닝**: Flag는 확률→라벨로의 비선형 맵핑이 필요. dev에서 F1을 직접 최대화하는 임계값 $[\tau_c]$을 선택하면, 테스트에서도 F1 기대치가 향상(ERM 원리).

    요약하면, epoch만 늘리면 empirical fit은 좋아지지만 다목적 dev 목표는 악화될 수 있다. dev 기반 합성 목적 최적화가 실제 목표(게임 상태 일관+대화 품질)에 더 직접적이다.  
    

---

## 추론 파이프라인

학습과 달리 추론에는 **Backward·Optimizer가 없고**, KV 캐시를 이용한
**증분 어텐션**이 핵심 차이이며, 추가 헤드는 <STATE> 고정 표현에서 단발로 계산된다.


- **1) 프롬프트 구성**  
  `<SYS>`, `<RAG>` 주입, `<STATE>`, `<NPC>`까지 구성 후 토크나이즈.

- **2) Forward (신경망 통과, KV 캐시)**  
  - $[t{=}1]$: 입력 시퀀스 전체 $Q,K,V$ 계산 → 최종 hidden state $H^{(L)}$ 산출.  
  - $[t{>}1]$: 새로운 토큰에 대한 $Q_t$만 계산하고, $K,V$는 캐시에 누산:  

    $[
        K \leftarrow \mathrm{concat}(K, K_t), \quad V \leftarrow \mathrm{concat}(V, V_t)
        ]$

    $[
        Z_t = \frac{Q_t K^\top}{\sqrt{d_k}},\quad P_t = \mathrm{softmax}(Z_t),\quad Y_t = P_t V
        ]$


      - **효과**: 이전 step의 $K,V$를 재활용해 $O(T^2)$ 어텐션을 $O(T)$로 줄여 추론 속도를 크게 향상.  
      - 학습 시 teacher forcing을 쓰던 것과 달리, 추론은 오직 과거 토큰의 캐시를 활용해 한 토큰씩 생성.

- **3) 추가 헤드 계산**  
  - 동일 forward에서 얻은 $H^{(L)}$ 중 `<STATE>` 위치 풀링 $h_*$로:
    

    $[
        \hat{\delta} = \tanh(W_\Delta h_*) \in [-1,1]^2,\quad
        \hat{p} = \sigma(W_F h_*) \in [0,1]^C
        ]$
      - $\hat{\delta}$는 trust·relationship 변화량,  
      $\hat{p}_c$는 각 flag의 발생 확률을 의미.

- **4) 응답 디코딩**
  - 다음 토큰 선택:  
  &emsp;Greedy: $\left(\mathrm{argmax}_y\, p_\theta(y|x,\langle s\rangle)\right)$  
  &emsp;Sampling: Top-$k$ / Nucleus($p$) 필터링 후 확률 샘플




  - 자기회귀적으로 반복.

- **5) 후처리**  
  - Delta: 정책에 따라 범위 clamp  
  - Flag: $\hat{y}_c = \mathbb{1}[\hat{p}_c \ge \tau_c]$  
    (임계값 $\tau_c$는 dev에서 F1 최대화로 추정)

---


# 🧱 학습 데이터 구조 설계
---

## 1. 시스템 개요와 설계 목표

### 1.1 목표

목표:
- 대화형 NPC 시스템에서 자연스러운 응답과 함께 수치적 상태 변화(delta)와 행동 플래그(flag)를 동시 산출.
  - LLM이 자연스러운 NPC 응답(Response)을 생성
  - 추가 헤드가 게임 상태 변화(Delta: 연속값)와 행동 플래그(Flag: 멀티라벨)도 동시 예측
- RAG를 통해 세계관 지식(Lore)과 상황 힌트(Description)를 프롬프트에 동적으로 주입하여 응답 품질(맥락·서사·감정)을 향상.
- 학습-추론 간 포맷 정합성 유지, 서버-게임 로직과 자연스럽게 연동
- 필터 기반 정책(postprocess)으로 하드 룰을 보정해 게임 상태의 일관성과 안정성 확보.

### 1.2 핵심 설계 원칙
- 병렬 멀티태스크: 자연어 생성(LM)과 수치 예측(delta, flag)을 동일 입력에 대해 동시에 학습.
- 위치 마커(State 토큰): delta/flag는 <STATE> 위치의 hidden state를 사용해 예측.
- 프롬프트 분할: <SYS> 고정 메타, <RAG> 동적 지식, <PLAYER_STATE>, <CTX>, <PLAYER>, <STATE>, <NPC>.
- 데이터 중심 결합: 구조적으로는 병렬이지만, “동일 상태 → 세 출력이 항상 짝지어 등장”하도록 데이터셋을 설계해 응답-수치가 맞물리도록 유도.
---

## 2. 데이터 스키마와 RAG 프롬프트 구성

### 2.1 학습 데이터(JSONL) 예시
- 학습에서는 lore/description을 비워 둔다(추론 시 RAG로 채움).
- 학습 데이터의 일부분(3%~5% 정도)은 lore/description을 채워서 추론 시에 ignore하는 상황을 막는다

```json
{
  "npc_id": "mother_abandoned_factory",
  "npc_location": "map1",
  "tags": {
    "quest_stage": "in_progress",
    "relationship": 0.35,
    "trust": 0.35,
    "npc_mood": "grief",
    "player_reputation": "helpful",
    "style": "emotional"
  },
  "lore": "이 공장은 수십 년 전 화재로 폐쇄되었다.",
  "description": "사진을 제시했고 공장을 방문했다면 신뢰 상승과 아이템 지급 가능.",
  "player_state": {
    "items": ["photo_forgotten_party"],
    "actions": ["visited_factory", "talked_to_guard"],
    "position": "factory_gate"
  },
  "context": [
    { "role": "player", "text": "사실 이 공장을 돌아다니면서..." },
    { "role": "npc", "text": "혹시 그 파티에 Jason도 있었나요..." }
  ],
  "player_utterance": "아! 머리가!!! 갑자기 기억이 떠올랐어요...",

  "response": "오 맙소사… Jason… 맞아요. 이렇게라도 그의 얼굴을 볼 수 있다니… 고마워요.",

  "delta": {
    "trust": 0.45,
    "relationship": 0.20
  },

  "flag": {
    "give_item": {
      "score": 0.43,
      "threshold": 0.65,
      "label": 0
    },
    "npc_main_story": {
      "score": 0.92,
      "threshold": 0.5,
      "label": 1
    },
    "quest_stage_change": {
      "score": 0.12,
      "threshold": 0.6,
      "label": 0
    },
    "set_waypoint_town": {
      "score": 0.0,
      "threshold": 0.5,
      "label": 0
    },
    "unlock_hidden_path": {
      "score": 0.0,
      "threshold": 0.5,
      "label": 0
    },
    "start_escort": {
      "score": 0.0,
      "threshold": 0.5,
      "label": 0
    },
    "end_conversation": {
      "score": 0.0,
      "threshold": 0.5,
      "label": 0
    }
  }
}

```
- 라벨 설계
  - Delta: `[trust_delta, relationship_delta] ∈ [-1,1]`  
  - Flag: 각 항목이 0.0–1.0 사이의 **연속값**(확률·신뢰도 라벨)  
    - 예: “give_item: 0.82, quest_stage_change: 0.61 …”
    - 학습 시 BCEWithLogitsLoss로 연속 타깃 학습(스무딩 라벨처럼 동작)  
    - 추론 시 점수 그대로 반환 → ai-server postprocess에서 RAG/threshold/mapping으로 텍스트/액션으로 해석

- ALL_FLAGS(인덱스 고정):  
  0 give_item, 1 npc_main_story, 2 quest_stage_change, 3 set_waypoint_town, 4 unlock_hidden_path, 5 start_escort, 6 end_conversation, 7 open_shop

- 추론 후 흐름(요약)  
  - 모델: flag 점수 배열(0..1) 출력  
  - ai-server postprocess: RAG/인덱스 매핑/임계치로 텍스트 변환  
  - game-server: index→flag_name 매핑 후 로직 실행

### 2.2 추론 프롬프트 레이아웃(학습과 동일한 섹션·순서)
- <SYS>: npc_id~style(고정 메타)
- <RAG>: lore(세계관) + description(조건·행동 힌트) — 추론 시 RAG로 채움
- <PLAYER_STATE>: items/actions/position
- <CTX>: 최근 대화
- <PLAYER>: 최신 발화
- <STATE>: 추가 헤드 pooling 기준 마커
- <NPC>: 모델 응답 시작

```python
def build_main_prompt(pre, session_id, npc_id):
    tags = pre.get("tags", {})
    ps = pre.get("player_state", {})
    lore_text = (pre.get("lore","") or "").strip() or "(없음)"
    desc_text = (pre.get("description","") or "").strip() or "(없음)"

    lines = []
    # SYS
    lines += [
        "<SYS>",
        f"NPC_ID={pre.get('npc_id','')}",
        f"NPC_LOCATION={pre.get('npc_location','')}",
        "TAGS:",
        f" quest_stage={tags.get('quest_stage','')}",
        f" relationship={tags.get('relationship','')}",
        f" trust={tags.get('trust','')}",
        f" npc_mood={tags.get('npc_mood','')}",
        f" player_reputation={tags.get('player_reputation','')}",
        f" style={tags.get('style','')}",
        "</SYS>"
    ]
    # RAG
    lines += [
        "<RAG>",
        f"LORE: {lore_text}",
        f"DESCRIPTION: {desc_text}",
        "</RAG>"
    ]
    # PLAYER_STATE
    lines += ["<PLAYER_STATE>"]
    if ps.get("items"):   lines.append(f"items={','.join(ps['items'])}")
    if ps.get("actions"): lines.append(f"actions={','.join(ps['actions'])}")
    if ps.get("position"):lines.append(f"position={ps['position']}")
    lines += ["</PLAYER_STATE>"]
    # CTX
    lines += ["<CTX>"]
    for h in pre.get("context", []):
        lines.append(f"{h['role']}: {h['text']}")
    lines += ["</CTX>"]
    # PLAYER → STATE → NPC
    lines += [
        f"<PLAYER>{pre.get('player_utterance','').rstrip()}",
        "<STATE>",
        "<NPC>"
    ]
    return "\n".join(lines)
```

### 2.3 RAG 문서 유형과 역할
- type=trigger_def: 하드 조건·정책(필터-only, postprocess용)
- type=description: 조건/행동 경향 힌트(필터-only, 프롬프트용)
- type=lore: 세계관·배경(쿼리+필터, 프롬프트용)
- type=rag_doc(main/fallback): 필요 시 예시 문구(쿼리+필터) — 본 구조에서는 필수 아님

```json
[
  {
    "id": "mother_abandoned_factory_in_progress_def",
    "type": "trigger_def",
    "npc_id": "mother_abandoned_factory",
    "quest_stage": "in_progress",
    "location": "map1",
    "trigger_definitions": {
      "required_text": ["기억", "사진"],
      "required_items": ["photo_forgotten_party"],
      "required_actions": ["visited_factory"],
      "emotion_threshold": { "sad": 0.2 }
    },
    "delta_policy": {
      "trust": { "min": -0.3, "max": 0.3, "per_turn_cap": 0.15 },
      "relationship": { "min": -0.5, "max": 0.5, "per_turn_cap": 0.25 }
    },
    "flag_policy": {
      "allowed": ["npc_main_story", "give_item"],
      "forbidden": ["quest_complete"]
    }
  },
  {
    "id": "lore_mother_factory_story",
    "type": "lore",
    "npc_id": "mother_abandoned_factory",
    "location": "map1",
    "quest_stage": "any",
    "content": "이 공장은 수십 년 전 화재로 폐쇄되었고, 실비아 가족의 상처가 남아 있다."
  },
  {
    "id": "desc_mother_factory_in_progress",
    "type": "description",
    "npc_id": "mother_abandoned_factory",
    "location": "map1",
    "quest_stage": "in_progress",
    "content": "플레이어가 사진을 보여주고 공장을 방문했다면, 신뢰가 크게 상승하고 실비아는 gold_necklace를 건넬 수 있다."
  }
]
```

---


# 📌 학습 코드 구현 설명
---
>이 코드는 **데이터 전처리 → Forward → Loss → Backward → Optimization → Validation**  
파이프라인을 구현하기 위해, 모델 초기화부터 전처리 유틸, 학습 루프까지의 전 과정을 포함합니다.



### 0. 모델 및 토크나이저 초기화
- HuggingFace `AutoTokenizer`로 Qwen2.5-3B-Instruct 토크나이저 로드
- 도메인 전용 special token(`<SYS>`, `<CTX>`, `<PLAYER>`, `<NPC>`, `<STATE>`, `<RAG>`, `<PLAYER_STATE>`) 추가
- `<STATE>` 토큰 ID(`STATE_ID`) 저장 → Delta/Flag head pooling 시 마스크로 사용
- pad token 미설정 시 eos token으로 대체, padding side는 `"right"`

---

### 0-1. QLoRA 베이스 모델 로드
- `BitsAndBytesConfig`로 4bit NF4 양자화 설정, bfloat16 연산
- `prepare_model_for_kbit_training()` 호출로 양자화 모델 학습 준비
- LoRA 설정(`LoraConfig`)에서 attention/FFN projection 계층을 target_modules로 지정
- LoRA 어댑터 적용 후 학습 파라미터 수 출력

---

### 0-2. 멀티태스크 헤드 추가
- `delta_head`: hidden_size → 2 (trust, relationship 변화량)
- `flag_head`: hidden_size → NUM_FLAGS (멀티라벨 행동 플래그)

---

### 0-3. 전처리 유틸 함수
- `_ctx()`: 대화 context를 `"role: text"` 형식으로 합침
- `_player_state_block()`: 플레이어 상태(items, actions, position)를 `<PLAYER_STATE>` 블록으로 변환
- `_flags_to_vec()`: flag 필드를 0/1 또는 확률 벡터로 변환 (dict/list 모두 처리)
- `_format()`:  
  - `<SYS>` 블록에 NPC 메타데이터 삽입  
  - `<RAG>` 블록에 lore/description 삽입  
  - `<CTX>` 블록에 대화 맥락 삽입  
  - `<PLAYER>` 발화와 `<STATE>` 토큰, `<NPC>` 시작 태그까지 prompt 구성  
  - delta 범위 [-1,1]로 클램프  
  - flag는 `_flags_to_vec()`로 변환  
  - answer는 response + eos token

---

### 1. 데이터셋 분할 및 텐서 변환
- `datasets` 라이브러리로 JSONL 데이터 로드 후 **train/test split**
- `_tok()`에서:
  - `prompt`와 `answer`를 토크나이즈 → `input_ids`, `attention_mask`, `labels` 생성
  - `<NPC>` 이전은 `labels=-100`으로 마스킹 (CE Loss 무시)
  - `<STATE>` 이후 라벨은 Delta/Flag 예측용 보조 타겟(`delta`, `flag`)으로 함께 반환
- `set_format(type="torch", columns=cols)`로 모델 입력용 텐서 구조 지정

---

### 2. 클래스 불균형 보정을 위한 `pos_weight` 계산
- 모든 train 샘플에서 `flag` 벡터만 스택 → 클래스별 양성 비율 $p_c$ 계산
- $pos\_weight_c = \frac{1-p_c}{p_c}$로 **BCEWithLogitsLoss**의 pos_weight 생성
- 이후 Flag Loss 계산 시 클래스별 불균형을 보정

---

### 3. Collator 정의
- DataLoader에서 배치 샘플을 하나의 텐서로 합치는 역할
- `"input_ids"`, `"attention_mask"`, `"labels"`, `"delta"`, `"flag"`를 각각 스택

---

### 4. Flag 전용 메트릭 계산 함수
- `_best_thresholds()`:
  - dev 세트에서 Flag F1 스코어를 최대화하는 클래스별 최적 threshold 탐색
- `_flag_metrics()`:
  - micro/macro F1, AUROC, AUPRC 계산
  - threshold 적용 후 성능 반환 → dev 성능이 향상되면 best_thresholds 갱신

---

### 5. `MultiHeadTrainer` 클래스
HuggingFace `Trainer`를 상속해, **다중 헤드 Loss 계산**과 **추가 메트릭 로직**을 구현

#### (1) `_state_hidden()`  
- 모델 출력 hidden state 중 `<STATE>` 토큰 위치만 마스킹 후 평균 풀링
- `<STATE>`가 없으면 fallback으로 마지막 토큰 hidden 사용
- 반환된 state representation은 Delta/Flag head의 입력

#### (2) `compute_loss()`  
- Forward → hidden state 추출
- **LM Loss**: CE(ignore_index=-100)  
- **Delta Loss**: HuberLoss(δ=0.1)  
- **Flag Loss**: BCEWithLogitsLoss(pos_weight=…)  
- 총 손실:  
  

\[
  L = \alpha_{LM}L_{LM} + \beta_\Delta L_\Delta + \gamma_{Flag}L_{Flag}
  \]


- `(total, outputs)` 또는 `total` 반환 (Trainer 내부에서 backward 수행)

#### (3) `evaluate()`  
- dev 세트에서 ppl, Δ-MAE/MSE/R², Flag micro/macro F1, AUROC, AUPRC 계산
- Composite 지표 = $-\log(\text{ppl}) + R^2_{\Delta} + \text{F1}_{macro,Flag}$
- Composite 최고치 시 `best_thresholds` 갱신

---

### 6. 학습 설정 (`TrainingArguments`)
- 배치 사이즈, gradient accumulation, epoch 수, 러닝레이트/스케줄(cosine), warmup, weight decay, max_grad_norm 설정
- `gradient_checkpointing` 활성화로 메모리 절약
- `optim="paged_adamw_32bit"`로 QLoRA 환경에 최적화
- 조기 종료(`EarlyStoppingCallback`)로 과적합 방지

---

### 7. Trainer 인스턴스 생성 및 학습 실행
```python
trainer = MultiHeadTrainer(
    model=model,
    args=training_args,
    train_dataset=tok_train,
    eval_dataset=tok_dev,
    dev_dataset=tok_dev,
    data_collator=Collator(),
    flag_pos_weight=pos_weight,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)
trainer.train()
```


- `flag_pos_weight`와 `dev_dataset`을 커스텀 Trainer에 전달
- `train()` 호출 시:
  1. 전처리된 배치 입력
  2. Forward → Loss 계산
  3. Backward + Optimizer step
  4. 매 epoch마다 dev 성능 평가 → best 모델 저장

---

### 8. 학습 완료 후 Best Model 저장 및 업로드

학습이 끝나면 `load_best_model_at_end=True` 설정 덕분에 Trainer 객체는  
dev composite metric 기준으로 가장 성능이 좋은 모델(PEFT 어댑터)을 메모리에 로드한 상태가 된다.  
이 시점에서 다음 절차로 저장·업로드를 진행한다.

1. **로컬 저장**
   - `trainer.save_model(LOCAL_DIR)` : 베스트 LoRA 어댑터 가중치 저장
   - `tokenizer.save_pretrained(LOCAL_DIR)` : 토크나이저 저장
   - `delta_head.pt`, `flag_head.pt` : 멀티태스크 추가 헤드 가중치 저장
   - `flags.json` : ALL_FLAGS 순서 정보 저장
   - `thresholds.json` : dev 평가에서 찾은 per-class 최적 threshold 저장
   - `training_meta.json` : base 모델명, loss 가중치, 학습 하이퍼파라미터 등 메타데이터 저장
   - 체크포인트 폴더(`checkpoint-*`)와 `runs` 폴더는 업로드 전 정리 → 최종본만 남김

2. **HF Hub 업로드 – Feature 브랜치**
   - Hugging Face Hub에 로그인 후 `HfApi` 사용
   - 현재 저장소(`REPO_ID`)의 `feature/model-vK` 브랜치 목록을 조회해 가장 큰 K를 찾고, +1 한 새 브랜치명 생성
   - `api.upload_folder(..., revision=FEATURE_BRANCH)`로 LOCAL_DIR 전체를 해당 브랜치에 업로드
   - 이렇게 하면 학습된 모델이 버전 단위(`feature/model-v1`, `feature/model-v2`, …)로 아카이빙됨

3. **latest 브랜치 반영**
   - 현재는 `PUSH_TO_LATEST=True`로 설정해 학습 직후 latest 브랜치에도 동일한 파일을 업로드
   - 하지만 모델 규모·데이터셋이 커지면, latest 반영은 추론 검증 후 별도 셀에서 수행하는 것이 안전
     - Colab 노트북 구조:  
       **학습 코드 + 저장 코드(Feature 브랜치 업로드) → 추론 코드(검증) → latest 업로드 셀**
     - HF Spaces의 `hf-serve` 서버는 latest 브랜치에서 어댑터 파일을 불러와 추론 endpoint를 제공하므로,  
       latest 반영은 반드시 테스트 통과 후 진행

이 구조를 통해, 학습된 모델은 항상 버전별로 보존되고,  
latest 브랜치는 안정성이 검증된 모델만 반영되어 서비스에 사용된다.



In [ ]:
!pip list

Package                               Version
------------------------------------- -------------------
absl-py                               1.4.0
absolufy-imports                      0.3.1
accelerate                            1.10.1
aiofiles                              24.1.0
aiohappyeyeballs                      2.6.1
aiohttp                               3.12.15
aiosignal                             1.4.0
alabaster                             1.0.0
albucore                              0.0.24
albumentations                        2.0.8
ale-py                                0.11.2
altair                                5.5.0
annotated-types                       0.7.0
antlr4-python3-runtime                4.9.3
anyio                                 4.10.0
anywidget                             0.9.18
argon2-cffi                           25.1.0
argon2-cffi-bindings                  25.1.0
array_record                          0.8.1
arviz                                 0.22.0
astro

In [ ]:
# @title ====model struct & train====
# -*- coding: utf-8 -*-
import os, json, shutil, math, re, tempfile
import torch
import numpy as np
from datasets import Dataset
from torch import nn
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    Trainer, TrainingArguments
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    f1_score, roc_auc_score, average_precision_score
)

# ========= Config (파일 경로만 여기서 설정) =========
BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"
DATA_JSONL = "testcase.jsonl"
LOCAL_DIR = "/content/npc-lora-output"

MAX_LEN = 512
MICRO_BSZ = 1
GRAD_ACC = 8
EPOCHS = 5
LR = 3e-5
WARMUP_RATIO = 0.03

# ==== Loss weight config ====
WEIGHT_DECAY = 0.05
MAX_GRAD_NORM = 1.0
ALPHA_LM = 0.25
BETA_DELTA = 0.25
GAMMA_FLAG = 0.25
DELTA_THRESHOLD = 0.25
F1_THRESHOLD_GRID = [x/100.0 for x in range(5, 95, 5)]

# ========= 1) JSONL 안전 로딩 및 ALL_FLAGS 자동 추출 =========
data_list = []
bad_lines = 0
with open(DATA_JSONL, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        line = line.strip()
        if not line:
            continue
        try:
            obj = json.loads(line)
            data_list.append(obj)
        except json.JSONDecodeError as e:
            bad_lines += 1
            print(f"Skipping invalid JSON line {i}: {e}")

if len(data_list) == 0:
    raise RuntimeError("No valid records loaded from JSONL. Check DATA_JSONL path and format.")

# 자동으로 dataset에 존재하는 flag 키들 추출 (output.flag 기준)
flag_key_set = set()
for ex in data_list:
    out = ex.get("output", {})
    ff = out.get("flag", {})
    if isinstance(ff, dict):
        flag_key_set.update(ff.keys())
ALL_FLAGS = sorted(list(flag_key_set))
NUM_FLAGS = len(ALL_FLAGS)
print(f"Detected {NUM_FLAGS} flag keys: {ALL_FLAGS}")

# # ========= 2) pos_weight 계산 (원시 데이터에서 안전하게) =========
# # 사용할 라벨 키 순서를 ALL_FLAGS로 고정
# flags_list = []
# for ex in data_list:
#     f = ex.get("flag", {})
#     row = []
#     for k in ALL_FLAGS:
#         sub = f.get(k, {})
#         # try to get label in robust way (int or bool), fallback 0
#         lab = sub.get("label", None)
#         if lab is None:
#             # maybe there's no nested dict but a direct scalar (defensive)
#             if isinstance(sub, (int, float, bool)):
#                 lab = int(sub)
#             else:
#                 lab = 0
#         row.append(float(lab))
#     flags_list.append(row)

# flags_mat = torch.tensor(flags_list, dtype=torch.float32)  # shape (N, NUM_FLAGS)
# with torch.no_grad():
#     p = flags_mat.mean(dim=0).clamp(1e-3, 1-1e-3)
# pos_weight = ((1 - p) / p)  # tensor on CPU for now
# print("pos_weight (per-class):", pos_weight.tolist())

# ========= 3) Tokenizer & specials =========
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
SPECIALS = ["<SYS>", "<CTX>", "<PLAYER>", "<NPC>", "<STATE>", "<RAG>", "<PLAYER_STATE>"]
tokenizer.add_special_tokens({"additional_special_tokens": SPECIALS})
STATE_ID = tokenizer.convert_tokens_to_ids("<STATE>")

# ========= 4) Model (QLoRA 전처리 + PEFT Lora) =========
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    device_map="auto",
    trust_remote_code=True
)
base_model = prepare_model_for_kbit_training(base_model)
base_model.resize_token_embeddings(len(tokenizer))

peft_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.10, bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]
)
model = get_peft_model(base_model, peft_config)
model.print_trainable_parameters()

# ========= 5) 추가 헤드 생성 (크기는 NUM_FLAGS에 맞춤) =========
device = model.device
hidden_size = model.config.hidden_size
model.delta_head = nn.Linear(hidden_size, 2).to(device)
model.flag_head  = nn.Linear(hidden_size, NUM_FLAGS).to(device)
model.threshold_head = nn.Linear(hidden_size, NUM_FLAGS).to(device)

# ========= 6) 전처리/포맷 함수 (원본 데이터 구조 기반) =========
# ALL_FLAGS = [
#     "give_item",
#     "give_hint",
#     "change_npc_state",
#     "change_game_state",
#     "change_player_state",
#     "npc_action",
#     "unlock_hidden_path"
# ]

SYSTEM_TMPL = """<SYS>
NPC_ID: {npc_id}
LOCATION: {npc_location}
QUEST_STAGE: {quest_stage}
RELATIONSHIP: {relationship}
TRUST: {trust}
MOOD: {npc_mood}
PLAYER_REPUTATION: {player_reputation}
STYLE: {style}
</SYS>"""

def _ctx(ctx_list):
    if not ctx_list:
        return ""
    return "\n".join([f"{turn['role'].upper()}: {turn['text']}" for turn in ctx_list])

def _player_state_block(player_state):
    if not player_state:
        return ""
    lines = [f"{k.upper()}: {v}" for k, v in player_state.items()]
    return "<PLAYER_STATE>\n" + "\n".join(lines) + "\n</PLAYER_STATE>"

def _format_for_tokenizer(ex):
    inp = ex.get("input", {}) or {}
    out = ex.get("output", {}) or {}
    tags = inp.get("tags", {}) or {}

    npc_location = inp.get("npc_location", "")
    sys = SYSTEM_TMPL.format(
        npc_id=inp.get("npc_id", ""),
        npc_location=npc_location,
        quest_stage=tags.get("quest_stage", ""),
        relationship=tags.get("relationship", ""),
        trust=tags.get("trust", ""),
        npc_mood=tags.get("npc_mood", ""),
        player_reputation=tags.get("player_reputation", ""),
        style=tags.get("style", "")
    )

    lore_text = (inp.get("lore", "") or "").strip()
    rag_block = f"<RAG>\nLORE: {lore_text}\n</RAG>"

    ctx = _ctx(inp.get("context", []))
    player = (inp.get("player_utterance", "") or "").rstrip()
    answer_text = ((out.get("response", "") or "").rstrip()) + tokenizer.eos_token
    ps_block = _player_state_block(inp.get("player_state", {}))

    prompt = f"{sys}\n{rag_block}\n{ps_block}\n<CTX>\n{ctx}\n</CTX>\n<PLAYER>{player}\n<STATE>\n<NPC>"

    # --- delta 처리 ---
    delta_field = out.get("delta", {}) or {}
    d0 = float(delta_field.get("trust", 0.0))
    d1 = float(delta_field.get("relationship", 0.0))
    delta = [max(-1.0, min(1.0, d0)), max(-1.0, min(1.0, d1))]

    # --- flag 처리 ---
    flag_field = out.get("flag", {}) or {}
    flag_score, flag_threshold = [], []
    for name in ALL_FLAGS:
        sub = flag_field.get(name, {}) or {}
        score = float(sub.get("score", 0.0))
        threshold = float(sub.get("threshold", 0.5))
        flag_score.append(score)
        flag_threshold.append(threshold)

    return {
        "prompt": prompt,
        "answer": answer_text,
        "delta": delta,
        "flag": flag_score,
        "threshold": flag_threshold,
    }


# ========= 7) Dataset.from_list -> tokenize map =========
dataset = Dataset.from_list(data_list)
splits = dataset.train_test_split(test_size=0.15, seed=42)
train_ds = splits["train"]
test_ds  = splits["test"]

def tokenize_map(ex):
    fmt = _format_for_tokenizer(ex)
    p_ids = tokenizer(fmt["prompt"], truncation=True, max_length=MAX_LEN, add_special_tokens=False)["input_ids"]
    a_ids = tokenizer(fmt["answer"], truncation=True, max_length=MAX_LEN, add_special_tokens=False)["input_ids"]
    input_ids = (p_ids + a_ids)[:MAX_LEN]
    labels = ([-100]*len(p_ids) + a_ids)[:MAX_LEN]
    attention_mask = [1]*len(input_ids)
    pad = MAX_LEN - len(input_ids)
    if pad > 0:
        input_ids = input_ids + [tokenizer.pad_token_id]*pad
        attention_mask = attention_mask + [0]*pad
        labels = labels + [-100]*pad

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
        "delta": list(map(float, fmt["delta"])),
        "flag": list(map(float, fmt["flag"])),
        "threshold": list(map(float, fmt["threshold"])),
    }

tok_train = train_ds.map(tokenize_map, remove_columns=train_ds.column_names, batched=False)
tok_dev   = test_ds.map(tokenize_map, remove_columns=test_ds.column_names, batched=False)

# torch format으로 변환
cols = ["input_ids","attention_mask","labels","delta","flag","threshold"]
tok_train.set_format(type="torch", columns=cols)
tok_dev.set_format(type="torch", columns=cols)

# ========= 8) Data collator (stacks tensors) =========
class Collator:
    def __call__(self, batch):
        return {k: torch.stack([b[k] for b in batch]) for k in cols}

# # ========= 9) Flag metrics helpers (현재 사용 x) =========
# def _best_thresholds(y_true, y_prob, grid):
#     C = y_true.shape[1]
#     th = [0.5]*C
#     for c in range(C):
#         best_f1, best_t = 0.0, 0.5
#         t_true = y_true[:, c]
#         p_prob = y_prob[:, c]
#         for t in grid:
#             f1 = f1_score(t_true, (p_prob>=t).astype(int), zero_division=0)
#             if f1 > best_f1:
#                 best_f1, best_t = f1, t
#         th[c] = best_t
#     return th

# def _flag_metrics(y_true, y_prob):
#     y_true_np = y_true.cpu().numpy()
#     y_prob_np = y_prob.cpu().numpy()
#     th = _best_thresholds(y_true_np, y_prob_np, F1_THRESHOLD_GRID)
#     y_pred = (y_prob_np >= np.array(th)[None,:]).astype(int)
#     micro_f1 = f1_score(y_true_np, y_pred, average="micro", zero_division=0)
#     macro_f1 = f1_score(y_true_np, y_pred, average="macro", zero_division=0)
#     try: auroc_macro = roc_auc_score(y_true_np, y_prob_np, average="macro")
#     except: auroc_macro = float("nan")
#     try: auprc_macro = average_precision_score(y_true_np, y_prob_np, average="macro")
#     except: auprc_macro = float("nan")
#     return {"micro_f1": micro_f1, "macro_f1": macro_f1, "auroc_macro": auroc_macro, "auprc_macro": auprc_macro, "thresholds": th}

# ========= 10) Trainer subclass (flag/threshold 회귀 버전) =========
class MultiHeadTrainer(Trainer):
    def __init__(self, dev_dataset=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.dev_dataset = dev_dataset
        self.best_composite = -1e9

    def _state_hidden(self, outputs, inputs):
        h = outputs.hidden_states[-1]            # [B, T, H]
        ids = inputs["input_ids"]                # [B, T]
        mask = (ids == STATE_ID).unsqueeze(-1)   # [B, T, 1]
        counts = mask.sum(dim=1).clamp_min(1)    # [B, 1]
        pooled = (h * mask).sum(dim=1) / counts  # [B, H]
        fallback = h[:, -1, :]                   # [B, H]
        has_state = (counts.squeeze(-1) > 0).squeeze(-1)  # [B]
        return torch.where(has_state.unsqueeze(-1), pooled, fallback)

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels     = inputs.pop("labels")
        delta      = inputs.pop("delta").to(model.device)       # [B, 2]
        flag_score = inputs.pop("flag").to(model.device)        # [B, NUM_FLAGS]
        threshold  = inputs.pop("threshold").to(model.device)   # [B, NUM_FLAGS]

        outputs = model(**inputs, output_hidden_states=True)
        logits = outputs.logits

        lm_loss = nn.CrossEntropyLoss(ignore_index=-100)(
            logits.view(-1, logits.size(-1)),
            labels.view(-1)
        )

        st_hid = self._state_hidden(outputs, inputs)

        delta_pred = torch.tanh(model.delta_head(st_hid))                 # [-1,1]
        delta_loss = torch.nn.functional.huber_loss(delta_pred, delta, delta=0.1)

        flag_pred = torch.sigmoid(model.flag_head(st_hid))                # [0,1]
        flag_loss = nn.MSELoss()(flag_pred, flag_score)

        threshold_pred = torch.sigmoid(model.threshold_head(st_hid))      # [0,1]
        threshold_loss = nn.MSELoss()(threshold_pred, threshold)

        total = (
            ALPHA_LM * lm_loss
            + BETA_DELTA * delta_loss
            + GAMMA_FLAG * flag_loss
            + DELTA_THRESHOLD * threshold_loss
        )
        return (total, outputs) if return_outputs else total

    @torch.no_grad()
    def evaluate(self, eval_dataset=None, ignore_keys=None, metric_key_prefix="eval"):
        self.model.eval()
        eval_dataset = eval_dataset if eval_dataset is not None else self.dev_dataset
        dl = self.get_eval_dataloader(eval_dataset)

        total_tokens, total_lm_loss = 0, 0.0
        y_delta_pred, y_delta_tgt = [], []
        y_flag_pred, y_flag_tgt = [], []
        y_threshold_pred, y_threshold_true = [], []

        for batch in dl:
            batch = {k: v.to(self.model.device) for k, v in batch.items()}
            labels = batch["labels"]

            outputs = self.model(
                **{k: batch[k] for k in ["input_ids", "attention_mask"]},
                output_hidden_states=True
            )
            logits = outputs.logits
            ce = nn.CrossEntropyLoss(ignore_index=-100)(
                logits.view(-1, logits.size(-1)),
                labels.view(-1)
            )
            token_count = max((labels != -100).sum().item(), 1)
            total_lm_loss += ce.item() * token_count
            total_tokens += token_count

            st_hid = self._state_hidden(outputs, batch)

            y_delta_pred.append(torch.tanh(self.model.delta_head(st_hid)).cpu())
            y_delta_tgt.append(batch["delta"].cpu())

            y_flag_pred.append(torch.sigmoid(self.model.flag_head(st_hid)).cpu())
            y_flag_tgt.append(batch["flag"].cpu())

            y_threshold_pred.append(torch.sigmoid(self.model.threshold_head(st_hid)).cpu())
            y_threshold_true.append(batch["threshold"].cpu())

        # concat
        ppl = math.exp(total_lm_loss / max(total_tokens, 1))
        y_delta_pred = torch.cat(y_delta_pred); y_delta_tgt = torch.cat(y_delta_tgt)
        y_flag_pred = torch.cat(y_flag_pred); y_flag_tgt = torch.cat(y_flag_tgt)
        y_threshold_pred = torch.cat(y_threshold_pred); y_threshold_true = torch.cat(y_threshold_true)

        # 회귀 지표 계산
        delta_mae = mean_absolute_error(y_delta_tgt.numpy(), y_delta_pred.numpy())
        delta_mse = mean_squared_error(y_delta_tgt.numpy(), y_delta_pred.numpy())
        delta_r2  = r2_score(y_delta_tgt.numpy(), y_delta_pred.numpy())

        flag_mae = torch.mean(torch.abs(y_flag_pred - y_flag_tgt)).item()
        flag_mse = torch.mean((y_flag_pred - y_flag_tgt) ** 2).item()

        threshold_mae = torch.mean(torch.abs(y_threshold_pred - y_threshold_true)).item()
        threshold_mse = torch.mean((y_threshold_pred - y_threshold_true) ** 2).item()

        # composite: 언어모델 성능 + 회귀 품질의 간단한 합
        composite = -math.log(ppl + 1e-12) + float(delta_r2) + (1.0 - flag_mae) + (1.0 - threshold_mae)
        if composite > self.best_composite:
            self.best_composite = composite

        metrics = {
            f"{metric_key_prefix}_ppl": ppl,
            f"{metric_key_prefix}_delta_mae": float(delta_mae),
            f"{metric_key_prefix}_delta_mse": float(delta_mse),
            f"{metric_key_prefix}_delta_r2": float(delta_r2),
            f"{metric_key_prefix}_flag_mae": float(flag_mae),
            f"{metric_key_prefix}_flag_mse": float(flag_mse),
            f"{metric_key_prefix}_threshold_mae": float(threshold_mae),
            f"{metric_key_prefix}_threshold_mse": float(threshold_mse),
            f"{metric_key_prefix}_composite": float(composite)
        }
        self.log(metrics)
        return metrics

# ========= 12) Train arguments 설정 =========
from transformers import EarlyStoppingCallback

training_args = TrainingArguments(
    output_dir=LOCAL_DIR,
    per_device_train_batch_size=MICRO_BSZ,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACC,
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    max_grad_norm=MAX_GRAD_NORM,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_composite",
    greater_is_better=True,
    bf16=torch.cuda.is_available(),
    fp16=False,
    gradient_checkpointing=True,
    optim="adamw_torch",
    remove_unused_columns=False,
    report_to="none"
)

trainer = MultiHeadTrainer(
    model=model,
    args=training_args,
    train_dataset=tok_train,
    eval_dataset=tok_dev,
    dev_dataset=tok_dev,
    data_collator=Collator(),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

trainer.train()

# ========= 12) Train =========
trainer.train()


# ==== After trainer.train() ====
from huggingface_hub import notebook_login, HfApi

# 1) 저장 경로 정리 및 아티팩트 저장
os.makedirs(LOCAL_DIR, exist_ok=True)

# (1) 베스트 어댑터 저장: Trainer가 load_best_model_at_end=True면,
#     trainer.save_model이 "베스트" PEFT 어댑터를 저장
trainer.save_model(LOCAL_DIR)

# (2) 토크나이저 저장
tokenizer.save_pretrained(LOCAL_DIR)

# (3) 추가 헤드 저장
torch.save(model.delta_head.state_dict(), os.path.join(LOCAL_DIR, "delta_head.pt"))
torch.save(model.flag_head.state_dict(),  os.path.join(LOCAL_DIR, "flag_head.pt"))
torch.save(model.threshold_head.state_dict(), os.path.join(LOCAL_DIR, "threshold_head.pt"))


# (4) 플래그 순서/임계값/메타 저장
with open(os.path.join(LOCAL_DIR, "flags.json"), "w", encoding="utf-8") as f:
    json.dump({"ALL_FLAGS": ALL_FLAGS}, f, ensure_ascii=False, indent=2)

best_thresholds = getattr(trainer, "best_thresholds", None)
if best_thresholds is not None:
    with open(os.path.join(LOCAL_DIR, "thresholds.json"), "w", encoding="utf-8") as f:
        json.dump({"per_class_thresholds": best_thresholds}, f, ensure_ascii=False, indent=2)

with open(os.path.join(LOCAL_DIR, "training_meta.json"), "w", encoding="utf-8") as f:
    json.dump({
        "base_model": BASE_MODEL,
        "loss_weights": {"alpha_lm": ALPHA_LM, "beta_delta": BETA_DELTA, "gamma_flag": GAMMA_FLAG},
        "epochs": EPOCHS,
        "max_len": MAX_LEN,
        "grad_acc": GRAD_ACC,
        "micro_bsz": MICRO_BSZ
    }, f, ensure_ascii=False, indent=2)

# (5) 체크포인트/런 폴더 정리(로컬에선 남겨도 되지만 업로드 전엔 제거 추천)
for sub in os.listdir(LOCAL_DIR):
    path = os.path.join(LOCAL_DIR, sub)
    if sub.startswith("checkpoint-") or sub == "runs":
        shutil.rmtree(path, ignore_errors=True)
print("Local cleanup 완료 — 최종본만 남음")

# 2) HF Hub 업로드 (feature/model-vK → latest)
notebook_login()  # 입력한 토큰은 로그에 마스킹됨
api = HfApi()

REPO_ID = "m97j/npc_LoRA-fps"
LATEST_BRANCH = "latest"
FEATURE_PREFIX = "feature/model-v"

# feature 브랜치 자동 증가
branches = api.list_repo_refs(repo_id=REPO_ID).branches
nums = [int(re.search(rf"{FEATURE_PREFIX}(\d+)", b.name).group(1))
        for b in branches if re.match(rf"{FEATURE_PREFIX}\d+", b.name)]
next_num = max(nums) + 1 if nums else 1
FEATURE_BRANCH = f"{FEATURE_PREFIX}{next_num}"

api.create_repo(REPO_ID, repo_type="model", exist_ok=True)
try:
    api.create_branch(repo_id=REPO_ID, branch=FEATURE_BRANCH)
except Exception:
    pass

# 업로드(Feature)
api.upload_folder(
    folder_path=LOCAL_DIR,
    repo_id=REPO_ID,
    repo_type="model",
    path_in_repo=".",
    revision=FEATURE_BRANCH
)
print(f"Feature 업로드 완료: {FEATURE_BRANCH}")

Detected 7 flag keys: ['change_game_state', 'change_npc_state', 'change_player_state', 'give_hint', 'give_item', 'npc_action', 'unlock_hidden_path']


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

trainable params: 29,933,568 || all params: 3,115,331,584 || trainable%: 0.9608


Map:   0%|          | 0/4 [00:00<?, ? examples/s]

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

...

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_regression.py:1266: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:300: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_regression.py:1266: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:300: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/usr/local/lib/py

Epoch,Training Loss,Validation Loss


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_regression.py:1266: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:300: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_regression.py:1266: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:300: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/usr/local/lib/py

Local cleanup 완료 — 최종본만 남음


HfHubHTTPError: 401 Client Error: Unauthorized for url: https://huggingface.co/api/repos/create (Request ID: Root=1-68e14d72-1da24dcd4cc82f907826b02e;00d90c3d-f74f-404e-9fb9-bf727ad95e0b)

Invalid username or password.

## onnx export

In [ ]:
!pip install optimum[onnxruntime]


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 124.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 141.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 110.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.8/425.8 kB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 108.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 9.4 MB/s eta 0:00:00
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.1
    Uninstalling tokenizers-0.22.1:
      Successfully uninstalled tokenizers-0.22.1
  Attempting uninstall: transformers
    Found existing installation: transformers 4.56.2
    Uninstalling transformers-4.56.2:
      Successfully uninstalled transformers-4.56.2


In [ ]:
# (optional) install once after restart:
# !pip install -U transformers optimum[onnxruntime] peft huggingface_hub

from huggingface_hub import notebook_login, HfApi
from peft import PeftModel, PeftConfig
from transformers import AutoModelForCausalLM, AutoTokenizer
from pathlib import Path
import subprocess

# 0) paths
ADAPTER_DIR = "/content/testcase_output"        # LoRA 어댑터 출력 디렉토리
MERGED_DIR  = "/content/merged_for_onnx"        # 병합 + ONNX export용 디렉토리
Path(MERGED_DIR).mkdir(parents=True, exist_ok=True)

# 1) tokenizer (학습 당시 vocab 반영)
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR, trust_remote_code=True)
n_vocab = len(tokenizer)

# 2) base 모델 로드 후 vocab 크기 맞추기
peft_config = PeftConfig.from_pretrained(ADAPTER_DIR)
base_model = AutoModelForCausalLM.from_pretrained(
    peft_config.base_model_name_or_path,
    trust_remote_code=True
)
base_model.resize_token_embeddings(n_vocab)
base_model.config.vocab_size = n_vocab

# 3) LoRA 어댑터 병합
model_with_lora = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
merged_model = model_with_lora.merge_and_unload()
merged_model.config.vocab_size = n_vocab

# 4) 병합된 모델 저장
merged_model.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)
print("Merged model saved to:", MERGED_DIR)

# 5) ONNX export (CLI 사용)
#    --model: 병합된 모델 디렉토리
#    --task: causal-lm (텍스트 생성용)
#    마지막 인자: ONNX 파일 저장할 디렉토리
subprocess.run([
    "optimum-cli", "export", "onnx",
    "--model", MERGED_DIR,
    "--task", "causal-lm",
    MERGED_DIR
], check=True)
print("ONNX export complete. Files in:", MERGED_DIR)

# 6) Hugging Face Hub 업로드
notebook_login()
api = HfApi()

REPO_ID = "m97j/npc_LoRA-fps"
LATEST_BRANCH = "latest"

try:
    api.create_branch(repo_id=REPO_ID, branch=LATEST_BRANCH)
except Exception:
    pass

api.upload_folder(
    folder_path=MERGED_DIR,   # model.onnx + merged weights + config.json + tokenizer.json
    repo_id=REPO_ID,
    repo_type="model",
    path_in_repo=".",
    revision=LATEST_BRANCH
)
print(f"latest 브랜치 덮어쓰기 완료: {LATEST_BRANCH}")


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Merged model saved to: /content/merged_for_onnx
ONNX export complete. Files in: /content/merged_for_onnx


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...0001-of-00003.safetensors:   0%|          | 1.17MB / 4.98GB            

...

  ...0003-of-00003.safetensors:   1%|1         | 33.5MB / 2.43GB            

  ..._for_onnx/model.onnx_data:   0%|          | 6.63MB / 12.3GB            

  ...0002-of-00003.safetensors:   1%|          | 41.9MB / 4.93GB            

  ...erged_for_onnx/model.onnx:   3%|3         | 57.3kB / 1.65MB            

latest 브랜치 덮어쓰기 완료: latest


In [ ]:
from huggingface_hub import notebook_login, HfApi

# Hugging Face 로그인
notebook_login()
api = HfApi()

# 업로드할 로컬 디렉토리 (학습 결과 LoRA 어댑터 포함)

REPO_ID = "m97j/npc_LoRA-fps"
LATEST_BRANCH = "latest"

# 브랜치 생성 (이미 있으면 무시)
try:
    api.create_branch(repo_id=REPO_ID, branch=LATEST_BRANCH)
except Exception:
    pass

# 디렉토리 전체 업로드
api.upload_folder(
    folder_path="/content/testcase_output",   # 로컬 폴더
    repo_id=REPO_ID,
    repo_type="model",
    path_in_repo="testcase_output",           # 저장소 안에 이 이름으로 폴더 생성
    revision=LATEST_BRANCH
)

print(f"latest 브랜치 덮어쓰기 완료: {LATEST_BRANCH}")


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...0003-of-00003.safetensors:   1%|1         | 33.5MB / 2.43GB            

  ...adapter_model.safetensors:   1%|1         | 33.5MB / 2.60GB            

  ...0002-of-00003.safetensors:   1%|          | 25.1MB / 4.93GB            

...

  ...case_output/delta_head.pt:  43%|####2     | 7.85kB / 18.3kB            

  ...tcase_output/flag_head.pt:  43%|####2     | 25.4kB / 59.3kB            

  ..._output/threshold_head.pt:  43%|####2     | 25.5kB / 59.4kB            

  ..._output/training_args.bin:  43%|####2     | 2.48kB / 5.78kB            

latest 브랜치 덮어쓰기 완료: latest


### 추론 파이프라인(간략)

1. Prompt 생성 + RAG 삽입
2. Transformer forward
3. `<STATE>` pooling → Delta/Flag 예측
4. LM: `<NPC>` 이후 토큰 생성(causal)
5. 최종 결과 반환  
Backward·optimizer 단계 없음, 학습된 파라미터로 순전파만 수행.

---

In [ ]:
#@title ==== Inference Smoke Test ====
import os, json
import torch
from huggingface_hub import login, list_repo_refs, snapshot_download
from peft import PeftModel

REPO_ID = "m97j/npc-LoRA-fps"
FEATURE_PREFIX = "feature/model-v"

login()  # 토큰 마스킹

def get_latest_feature_branch(repo_id: str, prefix: str) -> str:
    refs = list_repo_refs(repo_id=repo_id)
    nums = []
    for b in refs.branches:
        name = b.name
        m = re.match(rf"{prefix}(\d+)", name)
        if m: nums.append((int(m.group(1)), name))
    if not nums:
        raise RuntimeError("No feature branches found")
    return sorted(nums)[-1][1]

# 1) 로컬 아티팩트 확인 or 최신 feature 스냅샷 다운로드
if not os.path.exists(LOCAL_DIR) or not os.path.exists(os.path.join(LOCAL_DIR, "adapter_config.json")):
    branch = get_latest_feature_branch(REPO_ID, FEATURE_PREFIX)
    cache_dir = snapshot_download(repo_id=REPO_ID, revision=branch, repo_type="model")
    LOCAL_DIR = cache_dir
    print(f"Downloaded snapshot from {branch} to {LOCAL_DIR}")
else:
    print(f"Using local artifacts at {LOCAL_DIR}")

# 2) 로드
tokenizer = AutoTokenizer.from_pretrained(LOCAL_DIR, use_fast=True, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32
)
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=bnb_config, device_map="auto", trust_remote_code=True
)

base.resize_token_embeddings(len(tokenizer))
model = PeftModel.from_pretrained(base, LOCAL_DIR, is_trainable=False)

device = model.device
hidden_size = model.config.hidden_size
model.delta_head = nn.Linear(hidden_size, 2).to(device)
model.flag_head  = nn.Linear(hidden_size, NUM_FLAGS).to(device)
model.flag_threshold_head = nn.Linear(hidden_size, NUM_FLAGS).to(device)  # ← 추가

model.delta_head.load_state_dict(torch.load(os.path.join(LOCAL_DIR, "delta_head.pt"), map_location=device))
model.flag_head.load_state_dict(torch.load(os.path.join(LOCAL_DIR, "flag_head.pt"),  map_location=device))
model.flag_threshold_head.load_state_dict(torch.load(os.path.join(LOCAL_DIR, "flag_threshold_head.pt"), map_location=device))  # ← 추가

# flags
flags_order = json.load(open(os.path.join(LOCAL_DIR, "flags.json"), "r", encoding="utf-8"))["ALL_FLAGS"]

# 3) 스모크 프롬프트
def build_prompt(pre):
    tags = pre["tags"]
    ps = pre["player_state"]
    lore = pre.get("lore",""); desc = pre.get("description","")
    lines = []
    lines += [
        "<SYS>",
        f"NPC_ID={pre['npc_id']}",
        f"NPC_LOCATION={pre['npc_location']}",
        "TAGS:",
        f" quest_stage={tags.get('quest_stage','')}",
        f" relationship={tags.get('relationship','')}",
        f" trust={tags.get('trust','')}",
        f" npc_mood={tags.get('npc_mood','')}",
        f" player_reputation={tags.get('player_reputation','')}",
        f" style={tags.get('style','')}",
        "</SYS>",
        "<RAG>",
        f"LORE: {lore}",
        f"DESCRIPTION: {desc}",
        "</RAG>",
        "<PLAYER_STATE>"
    ]
    if ps.get("items"):   lines.append(f"items={','.join(ps['items'])}")
    if ps.get("actions"): lines.append(f"actions={','.join(ps['actions'])}")
    if ps.get("position"):lines.append(f"position={ps['position']}")
    lines += ["</PLAYER_STATE>", "<CTX>"]
    for h in pre.get("context", []):
        lines.append(f"{h['role']}: {h['text']}")
    lines += ["</CTX>", f"<PLAYER>{pre.get('player_utterance','').rstrip()}", "<STATE>", "<NPC>"]
    return "\n".join(lines)

sample = {
  "npc_id": "mother_abandoned_factory",
  "npc_location": "map1",
  "tags": {"quest_stage":"in_progress","relationship":0.35,"trust":0.35,"npc_mood":"grief","player_reputation":"helpful","style":"emotional"},
  "lore": "이 공장은 수십 년 전 화재로 폐쇄되었다.",
  "description": "사진을 제시했고 공장을 방문했다면 신뢰 상승과 아이템 지급 가능.",
  "player_state": {"items":["photo_forgotten_party"],"actions":["visited_factory"],"position":"factory_gate"},
  "context": [{"role":"player","text":"사실 이 공장을 돌아다니면서..."},{"role":"npc","text":"혹시 그 파티에 Jason도 있었나요..."}],
  "player_utterance": "아! 머리가!!! 갑자기 기억이 떠올랐어요..."
}

prompt = build_prompt(sample)
inputs = tokenizer(prompt, return_tensors="pt").to(device)

with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=120, do_sample=False)
    gen = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

    # hidden states for heads
    outputs = model(**inputs, output_hidden_states=True)
    h = outputs.hidden_states[-1]
    STATE_ID = tokenizer.convert_tokens_to_ids("<STATE>")
    ids = inputs["input_ids"]
    mask = (ids == STATE_ID).unsqueeze(-1)
    counts = mask.sum(dim=1).clamp_min(1)
    pooled = (h * mask).sum(dim=1) / counts
    st_hid = pooled  # assume one <STATE>

    delta_pred = torch.tanh(model.delta_head(st_hid))[0].cpu().tolist()
    flag_prob  = torch.sigmoid(model.flag_head(st_hid))[0].cpu().tolist()
    flag_thr  = torch.sigmoid(model.flag_threshold_head(st_hid))[0].cpu().tolist()  # ← 추가

    flag_pred  = [int(p>=t) for p,t in zip(flag_prob, flag_thr)]

print("=== Response ===")
print(gen.strip())
print("\n=== Delta ===")
print({"trust_delta": delta_pred[0], "relationship_delta": delta_pred[1]})
print("\n=== Flags ===")
print({name: int(v) for name, v in zip(flags_order, flag_pred)})
print("\n=== Flag Prob ===")
print({name: round(v,3) for name, v in zip(flags_order, flag_prob)})



---

# 📦 **모델 배포 및 사용 안내**

### 1️⃣ 배포 개요
이 프로젝트는 QLoRA 기반 LoRA 어댑터 방식으로 미세조정된 **Qwen2.5-3B-Instruct** 디코더 모델에  
Delta/Flag 예측 헤드를 추가한 **멀티태스크 NPC 대화 모델**입니다.  
학습이 끝난 모델은 Hugging Face Hub에 업로드 후 `hf-serve` 또는 기타 Serving 환경에서 배포할 수 있습니다.

---

### 2️⃣ 모델 구성 요약
- **Base**: Qwen2.5-3B-Instruct (RoPE + GQA + RMSNorm + SwiGLU)
- **Adapter**: LoRA(QLoRA) — Attention/FFN projection에 rank-$r$ 저랭크 보정 적용
- **Heads**:
  - LM Head: `<NPC>` 이후 토큰 생성
  - Delta Head: <STATE> pooling → tanh → [-1, 1] 범위 연속값
  - Flag Head: <STATE> pooling → sigmoid → [0, 1] 멀티라벨 확률

---

### 3️⃣ 추론 엔드포인트 동작 방식
Serving 환경에서 엔드포인트 호출 시:
1. **입력**:  
   - `<SYS> ~ <NPC>`까지의 프롬프트 (RAG 주입 포함)  
   - `input_ids`, `attention_mask`
2. **Forward**:
   - Transformer 디코더가 causal attention으로 전체 hidden state $H^{(L)}$ 계산
   - `<STATE>` 위치 풀링하여 Delta/Flag 산출
   - `<NPC>` 이후 LM 토큰을 자기회귀 방식으로 생성
3. **출력**:
   ```json
   {
     "response": "<LM 생성 문장>",
     "delta": [trust_delta, relationship_delta],
     "flags": {"flag_name": bool, ...}
   }
   ```

---

### 4️⃣ Hugging Face Hub 배포
1. **모델 업로드**:
   ```bash
    api.upload_folder(
        folder_path=src_dir,
        repo_id=REPO_ID,
        repo_type="model",
        path_in_repo=".",
        revision=LATEST_BRANCH
    )
   ```
    아래의 코드쉘을 실행시에 hf-hub에서 가장 최신 버전 모델을 가져오거나, 학습 코드 실행 이후 런타임을 종료하지 않았다면 로컬 디렉토리의 새로운 최신 버전 모델을 latest branch에 upload한다. (기존 내용 덮어씌움)

2. **실제 hf spaces 의 inference endpoint에서 모델 사용 구조(hf-serve)**:
   ```bash
    from transformers import AutoModelForCausalLM
    from peft import PeftModel

    BASE_ID = "qwen/2.5-3B-instruct"
    ADAPTER_ID = "m97j/npc-LoRA-fps"

    base = AutoModelForCausalLM.from_pretrained(
        BASE_ID,
        device_map="auto",
        torch_dtype="auto"
    )

    # latest branch에서 adapter 불러오기
    model = PeftModel.from_pretrained(
        base,
        ADAPTER_ID,
        revision="latest",  # 브랜치 고정
        device_map="auto"
    )

   ```
    hugging face spaces에서 배포된 main model inference 작업 및 외부 요청 endpoint를 구현한 hf-serve/ 에 revision을 "latest"로 고정하여 배포 및 실제 사용 환경에서 main model adapter files를 현재 프로젝트 hf hub repo의 latest branch의 파일들만 로드하게 구성하여 실제로 모델 배포는 latest branch에 배포만 하면 됨
---

### 5️⃣ API 사용 예시
```python
from huggingface_hub import InferenceClient

client = InferenceClient("<username>/<repo_name>")

prompt = """<SYS>...
<STATE>
<NPC>"""

result = client.text_generation(prompt, max_new_tokens=120)
print(result.generated_text)
```

---

### 6️⃣ 사용 시 유의사항
- Delta/Flag 출력은 게임 로직에 따라 **후처리 정책**(clamp, threshold) 적용 필요
- Serving 시 GPU 메모리 최소 16GB 이상 권장
- 추론 파라미터:
  - `temperature`: 응답 창의성 조절
  - `top_k`, `top_p`: 샘플링 다양성 제어
  - `max_new_tokens`: 최대 생성 길이 제한

---

In [ ]:
# @title ==== Promote feature → latest ====

import os, re, shutil
from huggingface_hub import login, HfApi, list_repo_refs, snapshot_download

REPO_ID = "m97j/npc-LoRA-fps"
LATEST_BRANCH = "latest"
FEATURE_PREFIX = "feature/model-v"

login()

def latest_feature_branch(repo_id: str, prefix: str) -> str:
    refs = list_repo_refs(repo_id=repo_id)
    nums = []
    for b in refs.branches:
        m = re.match(rf"{prefix}(\d+)", b.name)
        if m: nums.append((int(m.group(1)), b.name))
    if not nums: raise RuntimeError("No feature branches")
    return sorted(nums)[-1][1]

api = HfApi()
try:
    api.create_branch(repo_id=REPO_ID, branch=LATEST_BRANCH)
except Exception:
    pass

# 업로드 소스 폴더 결정: 로컬 or 스냅샷
src_dir = LOCAL_DIR if os.path.exists(os.path.join(LOCAL_DIR, "adapter_config.json")) else None
if not src_dir:
    feat = latest_feature_branch(REPO_ID, FEATURE_PREFIX)
    src_dir = snapshot_download(repo_id=REPO_ID, revision=feat, repo_type="model")
    print(f"Using snapshot from {feat}")

api.upload_folder(
    folder_path=src_dir,
    repo_id=REPO_ID,
    repo_type="model",
    path_in_repo=".",
    revision=LATEST_BRANCH
)

import requests

# latest 업로드 완료 후
print(f"Promoted to {LATEST_BRANCH} from {src_dir}")

# hf-serve Space URL
HF_SERVE_URL = "https://m97j-PersonaChatEngine.hf.space/api/ping_reload"

give ping
try:
    r = requests.post(HF_SERVE_URL, timeout=30)
    r.raise_for_status()
    print("Ping sent to hf-serve, reload triggered:", r.json())
except Exception as e:
    print("Failed to ping hf-serve:", e)



In [ ]:
# @title etc)
import importlib

# 학습 코드에서 쓰는 외부 패키지 목록
packages = [
    "torch",
    "numpy",
    "datasets",
    "transformers",
    "peft",
    "sklearn",
    "huggingface_hub"
]

for pkg in packages:
    spec = importlib.util.find_spec(pkg)
    if spec is None:
        print(f"{pkg:<20} ❌ 설치 안 됨")
    else:
        try:
            mod = importlib.import_module(pkg)
            ver = getattr(mod, "__version__", "버전 정보 없음")
            print(f"{pkg:<20} ✅ {ver}")
        except Exception as e:
            print(f"{pkg:<20} ⚠️ 설치됨 (버전 확인 실패: {e})")

torch                ✅ 2.8.0+cu126
numpy                ✅ 2.0.2
datasets             ✅ 4.0.0
transformers         ✅ 4.55.4
peft                 ✅ 0.17.1
sklearn              ✅ 1.6.1
huggingface_hub      ✅ 0.34.4


# End of Notebook  